In [1]:

# -*- coding: utf-8 -*-
"""
Extrai itens (Persun) de PDFs, monta planilha Excel e gera log de erros.
- Varre a pasta de PDFs
- Identifica blocos com palavras-chave (toldo, persiana)
- Extrai quantidade, valor unitário e total
- Salva em planilha "planilha_orcamentos_persun.xlsx"
- Gera "log_erros_extracao_persun.txt" quando necessário
"""

import fitz  # PyMuPDF
import re
import os
from datetime import datetime
from openpyxl import Workbook

# === CONFIGURAÇÕES ===
pasta_pdf = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\ABERTURA DE REQUISIÇÃO\LOJAS NOVAS\PERSUN\2025\pdfs_auto_persun"

# Dicionário de materiais Persun (keyword -> código material)
material_rules = {
    "toldo": "25992",
    "persiana": "26706"
}

data_hoje = datetime.today().strftime("%d%m%Y")

def clean_number(num_str):
    """Limpa strings numéricas no padrão BR e converte para float."""
    if not num_str:
        return None
    s = re.sub(r"[^\d,\.]", "", num_str)
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except Exception:
        return None

def normalizar_bloco(bloco: str) -> str:
    """Normaliza blocos para evitar duplicações (caixa baixa e espaços)."""
    return re.sub(r"\s+", " ", bloco.strip().lower())

# === CRIAÇÃO DA PLANILHA ===
wb = Workbook()
ws = wb.active
ws.title = "ORCAMENTOS_PERSUN"
ws["A1"] = "ORDEM"
ws["B1"] = "MATERIAL"
ws["H1"] = "QTDE"
ws["I1"] = "VALOR UNITARIO"
ws["J1"] = "TOTAL ITEM"
ws["M1"] = "DATA"
ws["P1"] = "ARQUIVO"

linha = 2
log_erros = []
textos_processados = set()

# === PROCESSAMENTO DOS PDFs ===
for nome_arquivo in os.listdir(pasta_pdf):
    if not nome_arquivo.lower().endswith(".pdf"):
        continue

    caminho_pdf = os.path.join(pasta_pdf, nome_arquivo)
    try:
        # Lê todo o texto do PDF (todas as páginas)
        with fitz.open(caminho_pdf) as pdf:
            texto = "".join(pagina.get_text("text") for pagina in pdf)

        # Evita processar textos duplicados
        if texto in textos_processados:
            continue
        textos_processados.add(texto)

        texto = texto.replace("\n", " ").replace("\r", " ")
        texto_lower = texto.lower()
        encontrado = False
        blocos_unicos = set()

        for chave, codigo in material_rules.items():
            if chave in texto_lower:
                # Captura blocos que contenham a palavra-chave e algum valor "R$ 999,99"
                padrao_item = re.findall(
                    rf"({chave}.*?R\$\s*[\d\.,]+)",
                    texto,
                    flags=re.IGNORECASE | re.DOTALL
                )

                for bloco in padrao_item:
                    bloco_normalizado = normalizar_bloco(bloco)
                    if bloco_normalizado in blocos_unicos:
                        continue
                    blocos_unicos.add(bloco_normalizado)

                    qtd = None
                    total_item = None
                    valor_unit = None

                    # Tenta capturar um padrão com qtd + unidade + total
                    m_qtd_total = re.search(
                        r"([\d\.,]+)\s*(m2|pç|un).*?R\$\s*([\d\.,]+)",
                        bloco,
                        re.IGNORECASE
                    )
                    if m_qtd_total:
                        qtd = clean_number(m_qtd_total.group(1))
                        total_item = clean_number(m_qtd_total.group(3))
                        # Primeiro valor após "R$" como possível unitário
                        valor_unit_match = re.search(r"R\$\s*([\d\.,]+)", bloco)
                        if valor_unit_match:
                            valor_unit = clean_number(valor_unit_match.group(1))
                        elif qtd and total_item:
                            valor_unit = total_item / qtd

                    # Fallback: tenta achar qtd e total separadamente
                    if not qtd or not total_item:
                        qtd_match = re.search(r"(\d+[,.]?\d*)\s*(pç|m2|un)", bloco, re.IGNORECASE)
                        if qtd_match:
                            qtd = clean_number(qtd_match.group(1))
                        m_total = re.search(r"R\$\s*([\d\.,]+)", bloco)
                        if m_total:
                            total_item = clean_number(m_total.group(1))
                        if qtd and total_item and not valor_unit:
                            valor_unit = total_item / qtd

                    # Regra específica para persiana (normalmente vem como item único)
                    if "persiana" in bloco.lower() or "persiana" in chave.lower():
                        qtd = 1
                        if total_item:
                            valor_unit = total_item
                        else:
                            m_any = re.search(r"R\$\s*([\d\.,]+)", bloco)
                            if m_any:
                                total_item = clean_number(m_any.group(1))
                                valor_unit = total_item

                    # Se conseguiu montar os campos, grava a linha
                    if qtd and (valor_unit or total_item):
                        ws[f"A{linha}"] = "F"
                        ws[f"B{linha}"] = codigo
                        ws[f"H{linha}"] = qtd
                        ws[f"I{linha}"] = round(valor_unit, 2) if valor_unit else ""
                        ws[f"J{linha}"] = round(total_item, 2) if total_item else ""
                        ws[f"M{linha}"] = data_hoje
                        ws[f"P{linha}"] = nome_arquivo
                        linha += 1
                        encontrado = True
                        # ✅ Limita a UMA linha por tipo de material por arquivo
                        break
                    else:
                        log_erros.append(
                            f"⚠️ Falha ao extrair {chave} em {nome_arquivo}\n{bloco[:400]}...\n"
                        )

        if not encontrado:
            log_erros.append(f"⚠️ Nenhum item reconhecido em {nome_arquivo}")

    except Exception as e:
        log_erros.append(f"❌ Erro ao processar {nome_arquivo}: {str(e)}")

# === SALVAR PLANILHA ===
caminho_planilha = os.path.join(pasta_pdf, "planilha_orcamentos_persun.xlsx")
wb.save(caminho_planilha)
print(f"✅ Planilha gerada com sucesso em: {caminho_planilha}")

# === SALVAR LOG DE ERROS ===
if log_erros:
    caminho_log = os.path.join(pasta_pdf, "log_erros_extracao_persun.txt")
    with open(caminho_log, "w", encoding="utf-8") as log_file:
        for linha_log in log_erros:
            log_file.write(linha_log + "\n")
    print(f"⚠️ Veja o log: {caminho_log}")


✅ Planilha gerada com sucesso em: C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\ABERTURA DE REQUISIÇÃO\LOJAS NOVAS\PERSUN\2025\pdfs_auto_persun\planilha_orcamentos_persun.xlsx
⚠️ Veja o log: C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\ABERTURA DE REQUISIÇÃO\LOJAS NOVAS\PERSUN\2025\pdfs_auto_persun\log_erros_extracao_persun.txt
